In [ ]:
!pip install ase

In [3]:
import numpy as np
from ase.io import read, write
from ase import Atoms

In [ ]:
def split_along_b_line(input_filename, output_prefix="hBN", shift=0.0):
  #  Разрез по прямой, параллельной вектору b, через середину по нормали n = cross(c, b).
   # shift -- сдвиг линии от середины в Å (положительный сдвигает линию в сторону 'right').

    atoms = read(input_filename, format="vasp")
    cell = atoms.get_cell()        # 3x3 array: a, b, c (строки)
    positions = atoms.get_positions()

    a_vec = np.asarray(cell[0])
    b_vec = np.asarray(cell[1])
    c_vec = np.asarray(cell[2])

    # единичные векторы
    b_unit = b_vec / np.linalg.norm(b_vec)
    c_unit = c_vec / np.linalg.norm(c_vec)

    # вектор нормали в плоскости (перпендикулярный b, лежащий в a-b плоскости)
    n = np.cross(c_unit, b_unit)
    n_norm = np.linalg.norm(n)
    if n_norm < 1e-8:
        raise ValueError("Не удалось получить нормаль n = cross(c,b). Проверьте cell (возможно c параллелен b).")
    n_unit = n / n_norm

    # проекции атомов на n
    s = positions @ n_unit    # scalar projection on n
    s_min, s_max = s.min(), s.max()
    s_mid = 0.5*(s_min + s_max) + shift

    # делим по середине s_mid
    left_idx  = np.where(s <= s_mid)[0]
    right_idx = np.where(s >  s_mid)[0]

    left_atoms  = atoms[left_idx].copy()
    right_atoms = atoms[right_idx].copy()

    # сохраним с той же ячейкой (чтобы в VESTA было видно взаиморасположение)
    left_atoms.set_cell(cell); left_atoms.set_pbc(atoms.get_pbc()); left_atoms.wrap()
    right_atoms.set_cell(cell); right_atoms.set_pbc(atoms.get_pbc()); right_atoms.wrap()

    write(f"{output_prefix}_left.vasp",  left_atoms,  format="vasp")
    write(f"{output_prefix}_right.vasp", right_atoms, format="vasp")

    # диагностический вывод
    print("Input:", input_filename)
    print("Cell vectors (Å):\n a=", a_vec, "\n b=", b_vec, "\n c=", c_vec)
    print(f"n_unit  = {n_unit}   (should lie in a-b plane, perpendicular to b)")
    print(f"s range = {s_min:.6f} .. {s_max:.6f}")
    print(f"s_mid   = {s_mid:.6f}   (shift={shift} Å)")
    print(f"Total atoms: {len(atoms)}")
    print(f"Left  atoms: {len(left_atoms)}")
    print(f"Right atoms: {len(right_atoms)}")

    return left_atoms, right_atoms

def rotate_grains(left_atoms, right_atoms, cell, angle_deg):
    #Вращает левое и правое зерно вокруг оси z на ±angle_deg.
    #Центры вращения: 0.25 и 0.75 по оси a исходной структуры.

    positions_left  = left_atoms.get_positions()
    positions_right = right_atoms.get_positions()

    a_vec = np.asarray(cell[0])

    # исходный диапазон a
    all_positions = np.vstack([positions_left, positions_right])
    proj_a = all_positions @ (a_vec / np.linalg.norm(a_vec))
    a_min, a_max = proj_a.min(), proj_a.max()

    # центры вращения вдоль a
    left_center_a  = a_min + 0.25*(a_max - a_min)
    right_center_a = a_min + 0.75*(a_max - a_min)

    # центр по y и z: среднее атомов каждого зерна
    left_center_yz  = positions_left[:,1:].mean(axis=0)
    right_center_yz = positions_right[:,1:].mean(axis=0)

    left_center  = np.array([left_center_a,  *left_center_yz])
    right_center = np.array([right_center_a, *right_center_yz])

    # матрицы вращения
    theta = np.deg2rad(angle_deg)
    R_pos = np.array([[ np.cos(theta), -np.sin(theta), 0],
                      [ np.sin(theta),  np.cos(theta), 0],
                      [0,0,1]])
    R_neg = np.array([[ np.cos(-theta), -np.sin(-theta), 0],
                      [ np.sin(-theta),  np.cos(-theta), 0],
                      [0,0,1]])

    # применяем вращение
    new_left_positions  = ((positions_left  - left_center)  @ R_pos.T)  + left_center
    new_right_positions = ((positions_right - right_center) @ R_neg.T) + right_center

    left_atoms.set_positions(new_left_positions)
    right_atoms.set_positions(new_right_positions)
    output_prefix = "hBN_rotated_"
    write(f"{output_prefix}_left.vasp",  left_atoms,  format="vasp")
    write(f"{output_prefix}_right.vasp", right_atoms, format="vasp")

    # объединяем зерна
    combined_atoms = left_atoms + right_atoms
    combined_atoms.set_cell(cell)
    combined_atoms.set_pbc(left_atoms.get_pbc())
    write("hBN_grains_rotated.vasp", combined_atoms, format="vasp")
    print("Файл с повернутыми зернами сохранён как 'hBN_grains_rotated.vasp'")

    return combined_atoms

def remove_close_atoms_pbc(atoms, cutoff=1.2):
    ## Удаляет атомы, находящиеся ближе, чем cutoff Å друг к другу,
   ## учитывая периодические граничные условия (PBC).

    positions = atoms.get_positions()
    cell = atoms.get_cell()
    pbc = atoms.get_pbc()

    num_atoms = len(atoms)
    to_delete = set()

    # матрица всех расстояний с учетом PBC
    dist_matrix = atoms.get_all_distances(mic=True)

    for i in range(num_atoms):
        if i in to_delete:
            continue
        for j in range(i+1, num_atoms):
            if j in to_delete:
                continue
            if dist_matrix[i,j] < cutoff:
                to_delete.add(j)

    if to_delete:
        mask = np.array([i not in to_delete for i in range(num_atoms)])
        atoms = atoms[mask]
        print(f"Удалено {len(to_delete)} слишком близких атомов с учётом PBC (cutoff = {cutoff} Å)")
    else:
        print("Слишком близких атомов не найдено с учётом PBC.")

    return atoms


if __name__ == "__main__":
    angle_deg = 4.0
    left_atoms, right_atoms = split_along_b_line("9_6_mono_hbn_ni_2_cart_b_vac.vasp", output_prefix="hBN", shift=0.0)
    cell = left_atoms.get_cell()
    combined_atoms = rotate_grains(left_atoms, right_atoms, cell, angle_deg=angle_deg)

    # удаляем слишком близкие атомы на границе
    cleaned_atoms = remove_close_atoms_pbc(combined_atoms, cutoff=1.2)
    write(f"9_6_hBN_grains_rotated_{angle_deg}_grad_clr.vasp", cleaned_atoms, format="vasp")


In [ ]:
# Добавление вакуума по X и Y
atoms = read("9_6_ni_hbn_ni_2_cart.vasp")
atoms.center(vacuum=10, axis=(0,1))  # добавить 10 Å вакуума по X и Y
write("9_6_ni_hbn_ni_2_cart_b_vac.vasp", atoms, format="vasp")

In [ ]:
def remove_atoms_by_type(infile, outfile, remove_symbols):
  # Удаляет атомы по типу (символу элемента) из VASP файла.

    atoms = read(infile)

    # логическая маска: True если атом надо оставить
    mask = [atom.symbol not in remove_symbols for atom in atoms]

    new_atoms = atoms[mask]

    write(outfile, new_atoms, format="vasp", vasp5=True)


remove_atoms_by_type("9_6_ni_hbn_ni_2_cart_b_vac.vasp",
                     "9_6_3hbn_ni_2_cart_b_vac.vasp",
                     ["Ni"]) # тип атомов для удаления


In [ ]:
def remove_atoms_by_z(infile, outfile, z_value, tolerance=1e-3):
   # Удаляет атомы, у которых z-координата равна заданной (с допуском tol).
    atoms = read(infile)

    mask = [abs(atom.position[2] - z_value) > tolerance for atom in atoms]
    new_atoms = atoms[mask]

    write(outfile, new_atoms, format="vasp", vasp5=True)


remove_atoms_by_z("9_6_mono_hbn_ni_2_cart_b_vac.vasp",
                  "9_6_mono_hbn_ni_2_cart_b_vac.vasp",
                  z_value=14.13053,
                  tolerance=1e-0)


In [ ]:
def make_multilayer(infile, outfile, n_layers=2, dz=5.0):
   # Создаёт многослойную систему копированием плёнки по оси z.

    atoms = read(infile)
    all_atoms = atoms.copy()

    for i in range(1, n_layers):
        shifted = atoms.copy()
        shifted.positions[:, 2] += i * dz
        all_atoms += shifted

    # увеличить размер ячейки по z, чтобы вместить все слои
    cell = all_atoms.get_cell()
    cell[2, 2] += (n_layers - 1) * dz
    all_atoms.set_cell(cell, scale_atoms=False)

    write(outfile, all_atoms, format="vasp", vasp5=True)

make_multilayer("9_6_hBN_grains_rotated_4.0_gra_clr2.vasp",
                "9_6_3hBN_grains_rotated_4.0_gra_clr2.vasp",
                n_layers=3, #   Количество слоёв (>=1). Если 1 → просто сохранится исходный файл.
                dz=3.13) #Смещение следующего слоя вдоль оси z (в Å).


In [ ]:
def add_full_Ni_layers(hbn_file, ref_file, outfile, z_gap=2.0):
    # Добавляет верхний и нижний Ni-слои к деформированному hBN-сэндвичу,
    # масштабируя и центрируя Ni по активной области hBN (учитывает вакуум и зерна),
    # сохраняет многослойную структуру Ni.


    hbn = read(hbn_file)
    ref = read(ref_file)

    # активная область hBN по XY
    hbn_active = hbn[[atom.symbol in ("B", "N") for atom in hbn]]
    x_min, x_max = hbn_active.positions[:, 0].min(), hbn_active.positions[:, 0].max()
    y_min, y_max = hbn_active.positions[:, 1].min(), hbn_active.positions[:, 1].max()
    Lx_hbn, Ly_hbn = x_max - x_min, y_max - y_min
    x_center, y_center = 0.5*(x_max + x_min), 0.5*(y_max + y_min)

    # выбираем Ni
    ref_Ni = ref[[atom.symbol == "Ni" for atom in ref]]
    z_coords = ref_Ni.positions[:, 2]
    z_mid = 0.5*(z_coords.min() + z_coords.max())
    bottom_Ni = ref_Ni[z_coords < z_mid]
    top_Ni = ref_Ni[z_coords > z_mid]

    # размеры исходной Ni плёнки (для масштабирования XY)
    Lx_Ni = bottom_Ni.positions[:, 0].max() - bottom_Ni.positions[:, 0].min()
    Ly_Ni = bottom_Ni.positions[:, 1].max() - bottom_Ni.positions[:, 1].min()
    scale_x = Lx_hbn / Lx_Ni
    scale_y = Ly_hbn / Ly_Ni

    def scale_XY_layered(atoms, sx, sy, x_center_target, y_center_target):
        a = atoms.copy()
        # смещаем к центру плёнки по XY, масштабируем, потом центрируем под hBN
        mean_x = np.mean(a.positions[:, 0])
        mean_y = np.mean(a.positions[:, 1])
        a.positions[:, 0] = (a.positions[:, 0] - mean_x)*sx + x_center_target
        a.positions[:, 1] = (a.positions[:, 1] - mean_y)*sy + y_center_target
        return a

    bottom = scale_XY_layered(bottom_Ni, scale_x, scale_y, x_center, y_center)
    top = scale_XY_layered(top_Ni, scale_x, scale_y, x_center, y_center)

    # смещаем по Z
    z_min_hbn, z_max_hbn = hbn.positions[:, 2].min(), hbn.positions[:, 2].max()
    bottom.positions[:, 2] += z_min_hbn - bottom.positions[:, 2].max() - z_gap
    top.positions[:, 2] += z_max_hbn - top.positions[:, 2].min() + z_gap

    # объединяем все атомы
    total = hbn + bottom + top
    total.set_cell(hbn.get_cell(), scale_atoms=False)

    write(outfile, total, format="vasp", vasp5=True)

add_full_Ni_layers("9_6_3hBN_grns_rotated_4.0_gra_b_vac.vasp",
                     "9_6_ni_hbn_ni_2_cart.vasp",
                     "Ni_3hBN_Ni_new3.vasp",
                     z_gap=2.05) # Зазор между hBN и Ni-слоями (Å).